# **Image Classification Cat, Dog, and Other Model Deployment**

# **Nama : Doni Firmansyah**

In [1]:
# @markdown **Import Library**

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing import image
import os
from google.colab import files
import pathlib

In [2]:
# @markdown **Collecting Data** :\
# @markdown Dataset yang digunakan adalah dari https://www.kaggle.com/datasets/dhondoi/cat-dog-other

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d dhondoi/cat-dog-other -p pet_animal --unzip

100% 317M/318M [00:03<00:00, 68.1MB/s]
100% 318M/318M [00:04<00:00, 83.1MB/s]


In [3]:
# @markdown **Collecting Data** :\
# @markdown Checking Total Dataset

ANIMAL = []

print("---------------")
print("Before Splitting")

DIR = "pet_animal"
TOTAL_OUTPUT = len(os.listdir(DIR))
totalAll = 0

for labelName in os.listdir(DIR):
  ANIMAL.append(labelName)

  dirLabel = os.path.join(DIR, labelName)
  total = len([name for name in os.listdir(dirLabel) if os.path.isfile(os.path.join(dirLabel, name))])

  print(f"Count Image {labelName} : {total}")

  totalAll+=total

print(f"\nTotal Image : {totalAll}")
print(f"Label Names : {ANIMAL}")
print("---------------")

---------------
Before Splitting
Count Image cat : 5000
Count Image other : 5000
Count Image dog : 5000

Total Image : 15000
Label Names : ['cat', 'other', 'dog']
---------------


In [ ]:
# @markdown **Collecting Data** :\
# @markdown Checking Total Dataset

# TRANSLATE = {
#     "cane": "dog",
#     "cavallo": "horse",
#     "elefante": "elephant",
#     "farfalla": "butterfly",
#     "gallina": "chicken",
#     "gatto": "cat",
#     "mucca": "cow",
#     "pecora": "sheep",
#     "ragno": "spider",
#     "scoiattolo": "squirrel"}

# ANIMAL_DICT = []

# DIR = "raw-img"
# TOTAL_OUTPUT = len(os.listdir(DIR))

# i = 0
# totalAll = 0

# print("Before Splitting")
# for labelName in os.listdir(DIR):
#   if labelName in TRANSLATE.keys():
#     ANIMAL_DICT.append(TRANSLATE[labelName])
#   else:
#     ANIMAL_DICT.append(labelName)

#   dirLabel = os.path.join(DIR, labelName)
#   total = len([name for name in os.listdir(dirLabel) if os.path.isfile(os.path.join(dirLabel, name))])
#   print(f"Count Image {ANIMAL_DICT[i]} : {total}")
#   totalAll+=total
#   i+=1

# print("---------------")
# print(f"Total Image : {totalAll}")
# print("---------------")

Before Splitting
Count Image chicken : 3098
Count Image dog : 4863
Count Image cat : 1668
Count Image sheep : 1820
Count Image elephant : 1446
Count Image squirrel : 1862
Count Image horse : 2623
Count Image spider : 4821
Count Image butterfly : 2112
Count Image cow : 1866
---------------
Total Image : 26179
---------------


In [4]:
# @markdown **Pre-Processing Data** :\
# @markdown Image Augmentation

trainIDG = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.1,
    shear_range=0.2,
    fill_mode="nearest",
    validation_split=0.2
)
validationIDG = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

In [5]:
# @markdown **Pre-Processing Data** :\
# @markdown Creating Callback

class CustomCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):
        if logs.get("accuracy") > 0.92 and logs.get("val_accuracy") > 0.92:
            print(f"Training and Validation on epoch {epoch+1} get minimum accuracy (92%) : Train {logs.get('accuracy'):.2f} Validation {logs.get('val_accuracy'):.2f}")
            self.model.stop_training = True

In [10]:
# @markdown **Pre-Processing Data** :\
# @markdown Modelling

WIDTH = 150
HEIGHT = 150

model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(WIDTH, HEIGHT, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(TOTAL_OUTPUT, activation='softmax')
])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 74, 74, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_6 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 17, 17, 128)      

In [11]:
# @markdown **Pre-Processing Data** :\
# @markdown Generate Train and Validation Dataset

BATCH_SIZE = 32

print("After Splitting Using Image Data Generator")
print("\nDataset Train")
trainDataset = trainIDG.flow_from_directory(DIR,
                                            class_mode='categorical',
                                            target_size=(WIDTH, HEIGHT),
                                            subset="training",
                                            batch_size=BATCH_SIZE,
                                            seed=42
                                            )

print("\nDataset Validation")
validationDataset = validationIDG.flow_from_directory(DIR,
                                                      class_mode='categorical',
                                                      target_size=(WIDTH, HEIGHT),
                                                      subset="validation",
                                                      batch_size=BATCH_SIZE,
                                                      seed=42
                                                      )
print("---------------")

After Splitting Using Image Data Generator

Dataset Train
Found 12000 images belonging to 3 classes.

Dataset Validation
Found 3000 images belonging to 3 classes.
---------------


In [12]:
# @markdown **Pre-Processing Data** :\
# @markdown Compiler for Model

LEARNING_RATE = 0.001

model.compile(loss='categorical_crossentropy',
              optimizer=tf.optimizers.Adam(learning_rate=LEARNING_RATE),
              metrics=['accuracy'])

In [13]:
# @markdown **Processing Data** :\
# @markdown Fitting

EPOCH = 100

history = model.fit(trainDataset,
                    epochs=EPOCH,
                    verbose=2,
                    validation_data=validationDataset,
                    callbacks=[CustomCallback()]
                    # steps_per_epoch=2,
                    # validation_steps=2
                    )

Epoch 1/100
375/375 - 718s - loss: 0.9016 - accuracy: 0.5404 - val_loss: 0.8839 - val_accuracy: 0.5593 - 718s/epoch - 2s/step
Epoch 2/100
375/375 - 644s - loss: 0.8078 - accuracy: 0.6103 - val_loss: 0.9170 - val_accuracy: 0.5903 - 644s/epoch - 2s/step
Epoch 3/100
375/375 - 646s - loss: 0.7631 - accuracy: 0.6458 - val_loss: 0.7930 - val_accuracy: 0.6630 - 646s/epoch - 2s/step


KeyboardInterrupt: 

In [ ]:
# @markdown **Post-Processing Data** :\
# @markdown Accuracy Visualization Result

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('ACCURACY')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# @markdown **Post-Processing Data** :\
# @markdown Loss Visualization Result

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('LOSS')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# @markdown **Post-Processing Data** :\
# @markdown Predict

# uploaded = files.upload()

# for fn in uploaded.keys():
#   path = fn
#   img = image.load_img(path, target_size =(150,150))
#   plt.imshow(img)
#   plt.show()
#   x = image.img_to_array(img)
#   x = np.expand_dims(x, axis=0)

#   images = np.vstack([x])
#   classes = model.predict(images, batch_size=BATCH_SIZE)

# numbClass = np.argmax(classes, axis=1)
# print('Predicted class: ', ANIMAL_DICT[numbClass[0]])

#   print(fn)
#   print(classes)
  # if classes[0,0]!=0:
  #   print('Dog')
  # else:
  #   print('Cat')

In [ ]:
# @markdown **Post-Processing Data** :\
# @markdown Saving Model

# exportDir = 'saved_cnn_model/'
# tf.saved_model.save(model, exportDir)


# converter = tf.lite.TFLiteConverter.from_saved_model(exportDir)
# tfLite = converter.convert()


# tfLiteFile = pathlib.Path('pet_animal.tflite')
# tfLiteFile.write_bytes(tfLite)